# Полносвязный перцептрон с нуля

Нейронная сеть - это функция с параметрами. Она принимает вектор чисел на входе и возвращает число, вектор чисел или класс на выходе. В отличие от обычной заранее выписанной формулы, нейронная сеть обычно содержит большое количество параметров, которые можно подбирать по данным. Именно поэтому одну и ту же архитектуру можно применять к разным задачам: меняются не правила вычисления, а конкретные значения весов и смещений.

В этом ноутбуке строится маленькая полносвязная нейронная сеть без ML-библиотек. Весь прямой проход написан вручную: через списки, циклы, функции и обычную арифметику. Такой способ полезен в начале изучения, потому что он убирает ощущение "черного ящика": сеть оказывается не магическим объектом, а последовательностью понятных вычислений.

Backpropagation здесь намеренно не используется. Сначала важно разобрать, **что именно вычисляет сеть**, из каких математических операций она состоит, почему слои можно рассматривать как композицию функций и откуда появляется способность приближать сложные зависимости.

## 1. Нейронная сеть как функция

Пусть объект описан числовым вектором:

```text
x = (x1, x2, ..., xn)
```

Нейронная сеть задает отображение:

```text
F: R^n -> R^m
```

То есть сеть берет `n` входных чисел и возвращает `m` выходных чисел. В этом смысле нейронная сеть является обычной математической функцией. Особенность состоит в том, что эта функция имеет специальную структуру: она составлена из слоев, а каждый слой выполняет похожую операцию над вектором.

Например, для бинарной классификации часто используют:

```text
F: R^n -> [0, 1]
```

Тогда выход можно интерпретировать как степень уверенности модели в классе `1`. Если выход близок к `0`, модель склоняется к первому варианту; если близок к `1`, ко второму. При этом сама сеть не "понимает" смысл классов в человеческом смысле. Она вычисляет число по заданной формуле.

Слово **параметрическая** означает, что внутри функции есть настраиваемые числа:

- **веса** `w` - коэффициенты при входных признаках;
- **смещения** `b` - свободные члены, которые сдвигают вычисление;
- иногда также говорят просто: параметры сети `theta`.

Если записать это компактно:

```text
prediction = F(x; theta)
```

Здесь `x` - данные, `theta` - все веса и смещения, `prediction` - ответ сети.

Важно отделять переменные от параметров. Вход `x` меняется от объекта к объекту: один день облачный, другой ясный; один пациент имеет одни показатели, другой - другие. Параметры `theta` фиксированы во время прямого прохода и задают поведение модели. Обучение, которое будет рассматриваться отдельно, как раз и состоит в подборе этих параметров.

В этом ноутбуке параметры будут заданы вручную. Это позволяет сосредоточиться на устройстве функции `F`, не смешивая его с вопросом о том, как именно параметры находятся.

## 2. Мини-датасет

Создадим игрушечную задачу: нужно решить, **брать ли зонт**.

У каждого дня есть три признака:

- `cloudiness` - облачность от `0.0` до `1.0`;
- `humidity` - влажность от `0.0` до `1.0`;
- `wind` - ветер от `0.0` до `1.0`.

Целевая переменная:

- `umbrella = 1` - зонт нужен;
- `umbrella = 0` - зонт не нужен.

Это не настоящая метеорологическая модель, а небольшой контролируемый пример. Он нужен для того, чтобы видеть все вычисления и не прятать идею сети за большим объемом данных. На реальных задачах признаков могут быть сотни или тысячи, но принцип остается тем же: объект превращается в числовой вектор, а модель вычисляет по нему ответ.

Обратите внимание, что все признаки уже приведены к диапазону от `0` до `1`. Это удобно, потому что веса разных признаков становятся проще сравнивать. Если один признак измеряется в долях, другой в тысячах, а третий в миллионах, то один только масштаб может начать доминировать в вычислениях. Поэтому нормализация и стандартизация признаков - важная часть подготовки данных.

In [2]:
dataset = [
    {"day": "A", "cloudiness": 0.10, "humidity": 0.20, "wind": 0.10, "umbrella": 0},
    {"day": "B", "cloudiness": 0.20, "humidity": 0.30, "wind": 0.80, "umbrella": 0},
    {"day": "C", "cloudiness": 0.40, "humidity": 0.50, "wind": 0.20, "umbrella": 0},
    {"day": "D", "cloudiness": 0.65, "humidity": 0.70, "wind": 0.20, "umbrella": 1},
    {"day": "E", "cloudiness": 0.80, "humidity": 0.60, "wind": 0.30, "umbrella": 1},
    {"day": "F", "cloudiness": 0.90, "humidity": 0.90, "wind": 0.70, "umbrella": 1},
    {"day": "G", "cloudiness": 0.30, "humidity": 0.85, "wind": 0.90, "umbrella": 1},
    {"day": "H", "cloudiness": 0.55, "humidity": 0.40, "wind": 0.10, "umbrella": 0},
]

def print_dataset(rows):
    header = f"{'day':<4} {'cloud':>7} {'humid':>7} {'wind':>7} {'target':>7}"
    print(header)
    print("-" * len(header))
    for row in rows:
        print(
            f"{row['day']:<4} "
            f"{row['cloudiness']:>7.2f} "
            f"{row['humidity']:>7.2f} "
            f"{row['wind']:>7.2f} "
            f"{row['umbrella']:>7}"
        )

print_dataset(dataset)

day    cloud   humid    wind  target
------------------------------------
A       0.10    0.20    0.10       0
B       0.20    0.30    0.80       0
C       0.40    0.50    0.20       0
D       0.65    0.70    0.20       1
E       0.80    0.60    0.30       1
F       0.90    0.90    0.70       1
G       0.30    0.85    0.90       1
H       0.55    0.40    0.10       0


### Признаки и вектор

Нейронная сеть обычно не работает со словарями напрямую. Ей удобнее получить **вектор чисел**.

Для одного дня:

```text
x = [cloudiness, humidity, wind]
```

Например, день `D`:

```text
x = [0.65, 0.70, 0.20]
```

Векторизация - один из центральных приемов машинного обучения. Реальный объект сначала описывается набором измеримых характеристик, затем эти характеристики упорядочиваются в вектор, а уже после этого модель работает с ним как с точкой в пространстве признаков.

Если признаков три, каждый объект можно представить точкой в трехмерном пространстве. Если признаков сто, объект становится точкой в стомерном пространстве. Человеку трудно представить такое пространство геометрически, но формулы со скалярными произведениями и матрицами работают одинаково при любом числе измерений.

In [3]:
def row_to_vector(row):
    return [row["cloudiness"], row["humidity"], row["wind"]]

example = dataset[3]
x = row_to_vector(example)

print("День:", example["day"])
print("Вектор признаков:", x)
print("Правильный ответ:", example["umbrella"])

День: D
Вектор признаков: [0.65, 0.7, 0.2]
Правильный ответ: 1


## 3. Один искусственный нейрон

Пусть входной вектор состоит из трех признаков:

```text
x = (x1, x2, x3)
```

У нейрона есть три веса и одно смещение:

```text
w = (w1, w2, w3),    b = const
```

Сначала нейрон считает **аффинное преобразование**:

```text
z = w1*x1 + w2*x2 + w3*x3 + b
```

То же самое через скалярное произведение:

```text
z = w · x + b
```

После этого применяется функция активации `sigma`:

```text
a = sigma(z) = sigma(w · x + b)
```

Где:

- `x` - входной вектор;
- `w` - вектор весов;
- `b` - смещение;
- `z` - значение до активации;
- `a` - выход нейрона после активации.

С математической точки зрения нейрон состоит из двух частей. Первая часть линейно комбинирует признаки: каждый признак получает свой коэффициент, затем результаты складываются. Вторая часть применяет нелинейное преобразование. Эта простая конструкция оказывается очень мощной, когда нейронов много и они соединены в слои.

Без активации нейрон был бы просто линейной формулой с дополнительным свободным членом. Активация позволяет строить нелинейные зависимости.

In [4]:
import math

def weighted_sum(inputs, weights, bias):
    total = bias
    for x_i, w_i in zip(inputs, weights):
        total += x_i * w_i
    return total

def step(z):
    return 1 if z >= 0 else 0

def sigmoid(z):
    return 1 / (1 + math.exp(-z))

def relu(z):
    return max(0, z)

inputs = [0.65, 0.70, 0.20]
weights = [3.0, 2.0, 0.5]
bias = -2.5

z = weighted_sum(inputs, weights, bias)

print("inputs =", inputs)
print("weights =", weights)
print("bias =", bias)
print("z =", round(z, 4))
print("step(z) =", step(z))
print("sigmoid(z) =", round(sigmoid(z), 4))
print("relu(z) =", round(relu(z), 4))

inputs = [0.65, 0.7, 0.2]
weights = [3.0, 2.0, 0.5]
bias = -2.5
z = 0.95
step(z) = 1
sigmoid(z) = 0.7211
relu(z) = 0.95


### Геометрический смысл весов и смещения

Если использовать ступенчатую активацию:

```text
step(z) = 1, если z >= 0
step(z) = 0, если z < 0
```

то нейрон делит пространство входов на две части:

```text
w · x + b = 0
```

Это уравнение задает границу решения.

Для двух признаков граница будет прямой:

```text
w1*x1 + w2*x2 + b = 0
```

Для трех признаков - плоскостью:

```text
w1*x1 + w2*x2 + w3*x3 + b = 0
```

В более высоких размерностях такую границу называют гиперплоскостью.

Вектор весов `w` задает ориентацию этой границы. Если изменить соотношение весов, граница повернется. Смещение `b` сдвигает границу, не меняя ее направления. Поэтому веса отвечают не только за "важность признаков", но и за геометрию разделения пространства.

Вес отвечает за направление и важность признака:

- положительный вес увеличивает `z`, когда признак растет;
- отрицательный вес уменьшает `z`, когда признак растет;
- вес около нуля почти выключает влияние признака.

Смещение `b` сдвигает границу решения. Если `b` сильно отрицательный, нейрону нужно больше положительного сигнала от признаков, чтобы активироваться.

Один нейрон со ступенчатой активацией способен провести только одну линейную границу. Этого достаточно для простых случаев, но недостаточно для сложных данных, где классы переплетены. Поэтому нейроны объединяются в слои: несколько простых границ вместе могут образовывать более сложные области.

In [5]:
def explain_neuron(inputs, weights, bias, activation):
    print("Подробный расчет нейрона")
    print("-" * 32)
    total = bias
    print(f"Начинаем со смещения bias = {bias:.3f}")
    for index, (x_i, w_i) in enumerate(zip(inputs, weights), start=1):
        contribution = x_i * w_i
        total += contribution
        print(f"x{index} * w{index} = {x_i:.3f} * {w_i:.3f} = {contribution:.3f}")
    print(f"z = {total:.3f}")
    print(f"activation(z) = {activation(total):.3f}")

explain_neuron(
    inputs=[0.65, 0.70, 0.20],
    weights=[3.0, 2.0, 0.5],
    bias=-2.5,
    activation=sigmoid,
)

Подробный расчет нейрона
--------------------------------
Начинаем со смещения bias = -2.500
x1 * w1 = 0.650 * 3.000 = 1.950
x2 * w2 = 0.700 * 2.000 = 1.400
x3 * w3 = 0.200 * 0.500 = 0.100
z = 0.950
activation(z) = 0.721


## 4. Почему нужна функция активации

Активация делает модель нелинейной.

Если слой без активации записать как:

```text
y = W*x + b
```

и затем поставить еще один линейный слой:

```text
u = V*y + c
```

то получится:

```text
u = V*(W*x + b) + c
u = (V*W)*x + (V*b + c)
```

Это снова одно аффинное преобразование. Значит, много линейных слоев без активаций можно свернуть в один линейный слой.

Поэтому между слоями добавляют нелинейную функцию:

```text
y = sigma(W*x + b)
```

Теперь следующему слою приходит уже нелинейно преобразованный вектор, и вся сеть может описывать гораздо более сложные зависимости.

Интуитивно активация меняет характер модели. Линейная модель может только взвешивать признаки и складывать их. Нелинейная модель может реагировать на комбинации признаков: например, "если одновременно высокая влажность и высокая облачность", а не просто "чем больше влажность, тем больше ответ". Именно такие взаимодействия признаков часто делают нейронные сети полезными.

На первом этапе достаточно знать три активации:

| Активация | Идея | Диапазон |
|---|---|---|
| `step` | жесткое решение 0 или 1 | `{0, 1}` |
| `sigmoid` | мягкая вероятность | `(0, 1)` |
| `ReLU` | пропускает положительные значения | `[0, +inf)` |

В выходном слое бинарного классификатора удобно использовать `sigmoid`, потому что ее результат лежит между `0` и `1`. В скрытых слоях часто используют ReLU и похожие функции, потому что они просты, быстро вычисляются и хорошо работают в глубоких сетях.

In [6]:
values = [-4, -2, -1, 0, 1, 2, 4]

print(f"{'z':>5} {'step':>7} {'sigmoid':>9} {'ReLU':>7}")
print("-" * 32)
for value in values:
    print(f"{value:>5.1f} {step(value):>7} {sigmoid(value):>9.4f} {relu(value):>7.1f}")

    z    step   sigmoid    ReLU
--------------------------------
 -4.0       0    0.0180     0.0
 -2.0       0    0.1192     0.0
 -1.0       0    0.2689     0.0
  0.0       1    0.5000     0.0
  1.0       1    0.7311     1.0
  2.0       1    0.8808     2.0
  4.0       1    0.9820     4.0


## 5. От нейрона к слою

Один нейрон возвращает одно число.

Слой из `k` нейронов возвращает вектор из `k` чисел:

```text
a = (a1, a2, ..., ak)
```

Если вход имеет размерность `n`, а в слое `k` нейронов, то у слоя есть:

```text
W - матрица размера k x n
b - вектор размера k
```

Формула слоя:

```text
z = W*x + b
a = sigma(z)
```

В развернутом виде для `j`-го нейрона:

```text
zj = wj1*x1 + wj2*x2 + ... + wjn*xn + bj
aj = sigma(zj)
```

Полносвязный слой называется полносвязным, потому что каждый входной признак соединен с каждым нейроном слоя. Если входов `n`, а нейронов `k`, то количество весов равно:

```text
n * k
```

И еще есть `k` смещений.

Содержательно слой можно понимать как набор детекторов. Каждый нейрон слоя реагирует на свой шаблон во входных данных. Один нейрон может быть чувствителен к высокой облачности, другой - к сочетанию ветра и влажности, третий - к признакам ясной погоды. В реальных сетях такие интерпретации не всегда очевидны, но математически идея остается такой: слой превращает исходные признаки в новое представление.

Это новое представление может быть удобнее для следующего слоя. Поэтому нейронная сеть не просто классифицирует исходные признаки, а постепенно строит промежуточные признаки, которые помогают решать задачу.

In [7]:
def neuron_forward(inputs, weights, bias, activation):
    z = weighted_sum(inputs, weights, bias)
    return activation(z)

def dense_layer_forward(inputs, layer_weights, layer_biases, activation):
    outputs = []
    for weights, bias in zip(layer_weights, layer_biases):
        output = neuron_forward(inputs, weights, bias, activation)
        outputs.append(output)
    return outputs

hidden_weights = [
    [5.0, 4.0, 0.0],   # нейрон 1: облачно + влажно
    [0.0, 1.0, 5.0],   # нейрон 2: влажно + сильный ветер
    [-6.0, -1.0, 0.0], # нейрон 3: скорее ясная погода
]
hidden_biases = [-4.0, -3.0, 3.0]

sample = row_to_vector(dataset[3])
hidden_outputs = dense_layer_forward(sample, hidden_weights, hidden_biases, sigmoid)

print("Вход:", sample)
print("Выходы скрытого слоя:")
for i, value in enumerate(hidden_outputs, start=1):
    print(f"h{i} = {value:.4f}")

Вход: [0.65, 0.7, 0.2]
Выходы скрытого слоя:
h1 = 0.8859
h2 = 0.2142
h3 = 0.1680


## 6. Полносвязный перцептрон как композиция функций

Многослойную полносвязную сеть можно записать как последовательность слоев.

Для сети с одним скрытым слоем:

```text
h = sigma(W1*x + b1)
y = phi(W2*h + b2)
```

Где:

- `x` - входной вектор;
- `W1`, `b1` - веса и смещения скрытого слоя;
- `h` - вектор скрытых признаков;
- `W2`, `b2` - веса и смещение выходного слоя;
- `phi` - активация выходного слоя.

Если подставить `h` во вторую формулу:

```text
y = phi(W2 * sigma(W1*x + b1) + b2)
```

Это уже не просто одна линейная формула, потому что внутри есть нелинейная `sigma`.

Для нашей задачи:

```text
[cloudiness, humidity, wind] -> [h1, h2, h3] -> umbrella_probability
```

Входной слой только хранит признаки. Скрытый слой строит промежуточное представление. Выходной слой превращает это представление в итоговую вероятность.

С точки зрения анализа функций такая сеть является композицией простых отображений. Первый слой переводит точку из исходного пространства признаков в пространство скрытых признаков. Второй слой берет уже это новое описание и строит ответ. Если слоев больше, то таких преобразований становится больше:

```text
x -> a1 -> a2 -> ... -> y
```

Именно композиция отличает глубокие модели от одной большой линейной формулы. Каждый слой может немного менять геометрию данных, делая задачу для следующего слоя проще.

## 7. Почему полносвязная сеть может приближать сложные функции

Полносвязные нейронные сети важны не только потому, что их удобно программировать. У них есть сильное математическое свойство: при достаточном числе нейронов они могут приближать очень широкий класс функций.

Одна из классических формулировок называется **теоремой универсальной аппроксимации**.

Неформально:

> Если функция `f` непрерывна на ограниченной замкнутой области, то полносвязная нейронная сеть с нелинейной активацией может приблизить `f` сколь угодно точно.

Более математически:

```text
Пусть f: K -> R непрерывна,
где K - компактное подмножество R^n.

Тогда для любого epsilon > 0 существует нейронная сеть F такая, что

|F(x) - f(x)| < epsilon

для всех x из K.
```

Здесь важно каждое слово. Область `K` должна быть компактной, то есть ограниченной и замкнутой. Функция `f` должна быть непрерывной. Число `epsilon` задает допустимую ошибку приближения. Если `epsilon` маленькое, требуется более точное приближение.

Интуитивно нейроны можно представить как простые строительные элементы функции. Один нейрон создает простую нелинейную форму. Несколько нейронов могут сложить более гибкую кривую или поверхность. Большое число нейронов позволяет приближать функции с большим количеством изгибов, переходов и локальных особенностей.

В одномерном случае это похоже на приближение сложной кривой набором простых кусочков. В многомерном случае вместо кривой появляется поверхность или гиперповерхность в пространстве признаков. Полносвязная сеть настраивает параметры так, чтобы эта поверхность проходила рядом с нужной зависимостью.

Что это означает:

- сеть может приблизить прямую, параболу, синусоиду и многие более сложные зависимости;
- точность приближения задается числом `epsilon`;
- чтобы сделать ошибку меньше, обычно требуется больше нейронов или более удобная архитектура;
- нелинейная активация принципиальна: без нее сеть не получает универсальной выразительности.

Что это **не** означает:

- теорема не говорит, что маленькая сеть справится с любой задачей;
- теорема не говорит, что обучение обязательно найдет нужные веса;
- теорема не гарантирует хорошую работу на данных вне области `K`;
- теорема не отменяет необходимость нормальных данных.

Поэтому теорема универсальной аппроксимации объясняет потенциальную выразительность полносвязных сетей, но не решает всю задачу машинного обучения. Она отвечает на вопрос "может ли такая функция существовать", но не отвечает полностью на вопрос "как быстро и надежно найти ее параметры по данным".

In [8]:
network = {
    "hidden_weights": [
        [5.0, 4.0, 0.0],
        [0.0, 1.0, 5.0],
        [-6.0, -1.0, 0.0],
    ],
    "hidden_biases": [-4.0, -3.0, 3.0],
    "output_weights": [[4.0, 1.5, -2.0]],
    "output_biases": [-1.9],
}

def network_forward(inputs, network):
    hidden = dense_layer_forward(
        inputs,
        network["hidden_weights"],
        network["hidden_biases"],
        sigmoid,
    )
    output = dense_layer_forward(
        hidden,
        network["output_weights"],
        network["output_biases"],
        sigmoid,
    )
    return output[0], hidden

probability, hidden = network_forward(row_to_vector(dataset[3]), network)

print("Скрытый слой:", [round(v, 4) for v in hidden])
print("Вероятность, что зонт нужен:", round(probability, 4))

Скрытый слой: [0.8859, 0.2142, 0.168]
Вероятность, что зонт нужен: 0.836


## 8. Forward pass по всему датасету

**Forward pass** - это вычисление значения функции `F(x; theta)` при уже заданных параметрах.

Для каждого объекта из датасета делаем:

```text
x -> h = sigma(W1*x + b1) -> y = sigmoid(W2*h + b2)
```

Получаем число `y` от `0` до `1`.

Для бинарной классификации это число превращается в класс через порог:

```text
prediction = 1, если y >= threshold
prediction = 0, если y < threshold
```

Порог `0.5` удобен, но не является законом. Если ложный пропуск опаснее ложной тревоги, порог можно уменьшить. Если ложная тревога слишком дорогая, порог можно увеличить.

Важный момент: forward pass не меняет сеть. Он только применяет уже существующую функцию к данным. Это похоже на подстановку значения в формулу. Если формула фиксирована, то для одного и того же входа она всегда вернет один и тот же выход.

При обучении ситуация другая: после вычисления ответа считается ошибка, а затем параметры корректируются. Но даже во время обучения прямой проход остается отдельной частью процесса: сначала сеть считает предсказание, потом уже это предсказание сравнивается с правильным ответом.

In [9]:
def predict_class(probability, threshold=0.5):
    return 1 if probability >= threshold else 0

def bar(value, width=20):
    filled = round(value * width)
    return "#" * filled + "." * (width - filled)

threshold = 0.5

print(f"{'day':<4} {'target':>6} {'prob':>8} {'pred':>6}  bar")
print("-" * 52)

correct = 0
for row in dataset:
    probability, hidden = network_forward(row_to_vector(row), network)
    prediction = predict_class(probability, threshold)
    correct += int(prediction == row["umbrella"])
    print(
        f"{row['day']:<4} "
        f"{row['umbrella']:>6} "
        f"{probability:>8.3f} "
        f"{prediction:>6}  "
        f"{bar(probability)}"
    )

accuracy = correct / len(dataset)
print("-" * 52)
print(f"Accuracy на игрушечном датасете: {accuracy:.3f}")

day  target     prob   pred  bar
----------------------------------------------------
A         0    0.035      0  #...................
B         0    0.143      0  ###.................
C         0    0.337      0  #######.............
D         1    0.836      1  #################...
E         1    0.884      1  ##################..
F         1    0.960      1  ###################.
G         1    0.758      1  ###############.....
H         0    0.487      0  ##########..........
----------------------------------------------------
Accuracy на игрушечном датасете: 1.000


## 9. Подробная трассировка одного примера

Рассмотрим один объект максимально подробно.

Возьмем день `G`:

```text
cloudiness = 0.30
humidity   = 0.85
wind       = 0.90
```

Человеческая интуиция: облачность не очень высокая, но влажность и ветер большие. Возможно, зонт пригодится.

Для сети это не рассуждение словами, а набор числовых операций. Сначала исходные признаки попадают в скрытый слой. Каждый скрытый нейрон считает свою взвешенную сумму и применяет активацию. Затем выходной нейрон получает уже не исходные признаки, а значения скрытых нейронов. Поэтому итоговое решение зависит не только от отдельных признаков, но и от того, какие промежуточные комбинации были построены скрытым слоем.

In [10]:
def trace_network(row, network):
    inputs = row_to_vector(row)
    print(f"День {row['day']}")
    print(f"Входы: cloudiness={inputs[0]:.2f}, humidity={inputs[1]:.2f}, wind={inputs[2]:.2f}")
    print()

    hidden_values = []
    for neuron_index, (weights, bias) in enumerate(
        zip(network["hidden_weights"], network["hidden_biases"]),
        start=1,
    ):
        z = weighted_sum(inputs, weights, bias)
        a = sigmoid(z)
        hidden_values.append(a)
        print(f"Скрытый нейрон h{neuron_index}")
        print(f"  weights = {weights}, bias = {bias}")
        print(f"  z = {z:.4f}")
        print(f"  sigmoid(z) = {a:.4f}")
        print()

    output_weights = network["output_weights"][0]
    output_bias = network["output_biases"][0]
    z_output = weighted_sum(hidden_values, output_weights, output_bias)
    probability = sigmoid(z_output)
    prediction = predict_class(probability)

    print("Выходной нейрон")
    print(f"  hidden = {[round(v, 4) for v in hidden_values]}")
    print(f"  weights = {output_weights}, bias = {output_bias}")
    print(f"  z = {z_output:.4f}")
    print(f"  sigmoid(z) = {probability:.4f}")
    print(f"  prediction = {prediction}")
    print(f"  target = {row['umbrella']}")

trace_network(dataset[6], network)

День G
Входы: cloudiness=0.30, humidity=0.85, wind=0.90

Скрытый нейрон h1
  weights = [5.0, 4.0, 0.0], bias = -4.0
  z = 0.9000
  sigmoid(z) = 0.7109

Скрытый нейрон h2
  weights = [0.0, 1.0, 5.0], bias = -3.0
  z = 2.3500
  sigmoid(z) = 0.9129

Скрытый нейрон h3
  weights = [-6.0, -1.0, 0.0], bias = 3.0
  z = 0.3500
  sigmoid(z) = 0.5866

Выходной нейрон
  hidden = [0.7109, 0.9129, 0.5866]
  weights = [4.0, 1.5, -2.0], bias = -1.9
  z = 1.1400
  sigmoid(z) = 0.7577
  prediction = 1
  target = 1


## 10. Размерности и количество параметров

Архитектура:

```text
3 входа -> 3 скрытых нейрона -> 1 выход
```

Скрытый слой:

```text
W1 имеет размер 3 x 3
b1 имеет размер 3
```

Значит:

```text
параметров в W1: 3 * 3 = 9
параметров в b1: 3
итого: 12
```

Выходной слой:

```text
W2 имеет размер 1 x 3
b2 имеет размер 1
```

Значит:

```text
параметров в W2: 1 * 3 = 3
параметров в b2: 1
итого: 4
```

Всего:

```text
12 + 4 = 16 параметров
```

Общая формула для полносвязного слоя:

```text
если входов n, а нейронов k, то параметров n*k + k
```

Или:

```text
k * (n + 1)
```

Плюс один внутри скобок соответствует смещению каждого нейрона.

Количество параметров быстро растет. Если входов `1000`, а в слое `500` нейронов, то только весов будет `1000 * 500 = 500000`, и еще `500` смещений. С одной стороны, большое число параметров дает модели гибкость. С другой стороны, оно требует больше данных, вычислений и аккуратности при обучении.

Именно поэтому архитектура сети является важным выбором. Слишком маленькая сеть может не выразить нужную зависимость. Слишком большая сеть может оказаться избыточной, медленной и склонной подстраиваться под шум в данных.

In [11]:
def count_layer_parameters(layer_weights, layer_biases):
    weight_count = 0
    for neuron_weights in layer_weights:
        weight_count += len(neuron_weights)
    bias_count = len(layer_biases)
    return weight_count + bias_count

hidden_parameter_count = count_layer_parameters(
    network["hidden_weights"],
    network["hidden_biases"],
)
output_parameter_count = count_layer_parameters(
    network["output_weights"],
    network["output_biases"],
)

print("Параметров в скрытом слое:", hidden_parameter_count)
print("Параметров в выходном слое:", output_parameter_count)
print("Всего параметров:", hidden_parameter_count + output_parameter_count)

Параметров в скрытом слое: 12
Параметров в выходном слое: 4
Всего параметров: 16


## 11. Архитектура, forward pass и обучение

Важно разделять три разных вопроса.

**Архитектура:**

```text
Какая функция задана по форме?
Сколько слоев, сколько нейронов, какие активации?
```

**Forward pass:**

```text
Как посчитать F(x; theta), если параметры theta уже известны?
```

**Обучение:**

```text
Как подобрать theta, чтобы F(x; theta) давала хорошие ответы?
```

В этом ноутбуке разобраны архитектура и forward pass. Параметры заданы вручную, поэтому сеть не обучается, а только демонстрирует механизм вычисления.

Такое разделение полезно методически и математически. Архитектура определяет семейство функций, из которого мы выбираем модель. Forward pass показывает, как конкретная функция из этого семейства вычисляет ответ. Обучение выбирает конкретные параметры внутри заданного семейства.

Если говорить кратко:

```text
архитектура задает форму;
параметры задают конкретную функцию;
forward pass применяет эту функцию;
обучение меняет параметры.
```

После понимания прямого прохода становится естественным следующий шаг: ввести функцию потерь, измерить ошибку предсказаний и обсудить, как изменение весов влияет на эту ошибку. Именно к этому приводит backpropagation, но сам backpropagation имеет смысл только после того, как ясно, что сеть вычисляет в прямом направлении.

## 12. Обратное распространение ошибки

До этого момента сеть рассматривалась как готовая функция: веса уже заданы, и мы просто считаем ответ. Такой расчет называется **прямым проходом** (*forward pass*):

```text
входные признаки -> скрытые нейроны -> выходной нейрон -> предсказание
```

Но при обучении веса не заданы заранее. Их нужно подобрать. Для этого сеть должна после ошибки ответить на очень практичный вопрос:

```text
какой вес надо немного увеличить,
какой вес надо немного уменьшить,
а какой почти не трогать?
```

Обратное распространение ошибки (*backpropagation*) как раз и дает такой ответ. Оно не требует от нейрона “понимать” задачу. Оно просто аккуратно считает производные.

Для понимания этого раздела достаточно помнить одну идею: **производная показывает, как изменится результат, если чуть-чуть изменить вход**.


### 12.1. Минимум математики: что здесь значит производная

Пусть есть функция:

```text
y = f(x)
```

Производная `f'(x)` отвечает на вопрос:

```text
если x чуть-чуть увеличить, что произойдет с y?
```

Для нас это будет читаться так:

```text
производная положительная -> при увеличении x значение y растет;
производная отрицательная -> при увеличении x значение y уменьшается;
производная около нуля    -> маленькое изменение x почти не меняет y.
```

В обучении нейросети роль `y` играет ошибка `loss`, а роль `x` играет конкретный вес. Поэтому производная по весу отвечает на очень понятный вопрос:

```text
если немного увеличить этот вес, ошибка станет больше или меньше?
```

Это и есть главный смысл градиента.


### Визуально: куда идет информация



Синие стрелки показывают, как в прямом проходе считаются значения нейронов. Красные стрелки показывают, как в обратном проходе считается влияние ошибки на параметры.


### 12.2. Функция потерь: как измерить ошибку

Сначала нужно превратить качество ответа сети в одно число. Это число называется **функцией потерь** или просто `loss`.

Пусть сеть решает задачу: брать зонт или не брать. Для одного дня у нас есть правильный ответ `y_true` и предсказание сети `y_pred`. В нашем примере `y_pred` - это число от 0 до 1, потому что на выходе стоит сигмоида.

Например:

```text
y_true = 1
y_pred = 0.76
```

Сеть в целом права, потому что вероятность больше 0.5. Но она не идеально уверена: правильный ответ 1, а сеть дала 0.76. Значит, ошибка все еще есть.

Для простоты возьмем квадратичную функцию потерь:

```text
loss = 0.5 * (y_pred - y_true)^2
```

Эту формулу можно читать почти буквально:

```text
y_pred - y_true        -> насколько промахнулась сеть;
(...)^2                -> делаем ошибку положительной;
0.5                    -> удобный множитель, чтобы производная была красивее.
```

Если `loss` большой, сеть ошибается сильно. Если `loss` маленький, сеть близка к правильному ответу. Обучение - это попытка менять веса так, чтобы `loss` становился меньше.


### 12.3. Градиент: куда двигать вес

Теперь представим один конкретный вес `w`. Он влияет на предсказание, а предсказание влияет на `loss`. Значит, можно мысленно считать, что ошибка зависит от этого веса:

```text
loss = loss(w)
```

Производная ошибки по весу записывается так:

```text
d_loss / d_w
```

Не нужно пугаться записи. Ее можно читать словами:

```text
как изменится loss, если немного изменить w?
```

Возможны три ситуации:

```text
d_loss / d_w > 0
если увеличить вес, ошибка увеличится;
значит, чтобы уменьшить ошибку, вес надо двигать вниз.

 d_loss / d_w < 0
если увеличить вес, ошибка уменьшится;
значит, вес надо двигать вверх.

 d_loss / d_w ≈ 0
около текущего значения этот вес почти не влияет на ошибку.
```

Поэтому базовое правило обновления веса выглядит так:

```text
new_weight = old_weight - learning_rate * gradient
```

Минус в формуле означает: мы идем не туда, где ошибка растет, а в противоположную сторону.

`learning_rate` - размер шага. Маленький шаг дает осторожное обучение. Слишком большой шаг может перескочить хорошее значение веса.


### 12.4. Что именно нужно посчитать в нашей сети

В этой лекции используется маленькая сеть:

```text
3 входа -> 3 скрытых нейрона -> 1 выходной нейрон
```

У нее есть веса и смещения:

```text
веса скрытого слоя:  от входов к h1, h2, h3;
bias скрытого слоя:  отдельное смещение для h1, h2, h3;
веса выхода:         от h1, h2, h3 к выходному нейрону;
bias выхода:         смещение выходного нейрона.
```

При обучении нужно получить производную для каждого параметра. То есть для каждого веса сеть должна понять:

```text
если чуть-чуть изменить именно этот вес,
как изменится итоговая ошибка?
```

Если параметров 16, как в нашей игрушечной сети, нужно 16 таких ответов. В больших сетях параметров гораздо больше, но принцип тот же.

Backpropagation нужен потому, что он не проверяет веса по одному грубой силой. Он использует уже сделанный forward pass и быстро считает все нужные производные.


### 12.5. Обозначения для одного объекта

Теперь введем обозначения. Они нужны не ради строгости, а чтобы было понятно, какую производную мы считаем.

Входы:

```text
x1 = cloudiness
x2 = humidity
x3 = wind
```

Каждый скрытый нейрон делает две операции.

Сначала считает сумму:

```text
z = входы * веса + bias
```

Потом применяет активацию:

```text
h = sigmoid(z)
```

Для первого скрытого нейрона это выглядит так:

```text
z_h1 = x1*w_11 + x2*w_12 + x3*w_13 + b_h1
h1 = sigmoid(z_h1)
```

Для остальных скрытых нейронов то же самое. Выходной нейрон получает уже не исходные признаки, а значения скрытых нейронов:

```text
z_out = h1*v1 + h2*v2 + h3*v3 + b_out
y_pred = sigmoid(z_out)
```

Здесь `v1`, `v2`, `v3` - веса от скрытых нейронов к выходу.


### 12.6. Правило цепочки: почему производные перемножаются

Главная трудность: вес в начале сети не влияет на ошибку напрямую. Он влияет на скрытый нейрон, скрытый нейрон влияет на выход, выход влияет на ошибку.

То есть получается цепочка:

```text
вес -> скрытый нейрон -> выход -> loss
```

Если каждое звено цепочки немного меняет следующее, то общий эффект получается как произведение этих маленьких эффектов. Это и есть правило цепочки.

Простой пример без нейросетей:

```text
если A влияет на B в 2 раза,
а B влияет на C в 3 раза,
то A влияет на C в 2 * 3 = 6 раз.
```

В нейросети вместо слов “в 2 раза” и “в 3 раза” стоят производные.

Например, выходной вес `v1` влияет на loss по цепочке:

```text
v1 -> z_out -> y_pred -> loss
```

Значит:

```text
d_loss/d_v1 = d_loss/d_y_pred * d_y_pred/d_z_out * d_z_out/d_v1
```

Эта формула выглядит длинно, но каждый кусок простой:

```text
d_loss/d_y_pred       -> как ошибка реагирует на предсказание;
d_y_pred/d_z_out      -> как сигмоида реагирует на входную сумму;
d_z_out/d_v1          -> как сумма выхода реагирует на вес v1.
```

Именно поэтому backpropagation идет справа налево: сначала мы знаем ошибку на выходе, потом постепенно раскладываем ее на предыдущие звенья.


<!-- derivative-friendly-extra -->
### Локальные производные, которые понадобятся

В этой маленькой сети нужны всего несколько простых производных.

```text
loss = 0.5 * (y_pred - y_true)^2
производная: d_loss/d_y_pred = y_pred - y_true
```

```text
y_pred = sigmoid(z_out)
производная: d_y_pred/d_z_out = sigmoid(z_out) * (1 - sigmoid(z_out))
```

```text
z_out = h1*v1 + h2*v2 + h3*v3 + b_out
производная по v1: d_z_out/d_v1 = h1
производная по b_out: d_z_out/d_b_out = 1
```

Последняя строка особенно важна: если сумма содержит `h1*v1`, то при изменении `v1` скорость изменения суммы равна `h1`. Это обычная производная линейной функции.


### 12.7. Как ошибка доходит до скрытого слоя

Для выходного слоя все довольно близко к ошибке: выходные веса стоят почти рядом с `loss`.

Для скрытого слоя путь длиннее. Например, вес `w_11` влияет на ошибку так:

```text
w_11 -> z_h1 -> h1 -> z_out -> y_pred -> loss
```

Поэтому производная раскладывается на несколько множителей:

```text
d_loss/d_w_11 = d_loss/d_y_pred
                * d_y_pred/d_z_out
                * d_z_out/d_h1
                * d_h1/d_z_h1
                * d_z_h1/d_w_11
```

Смысл каждого множителя:

```text
d_loss/d_y_pred   -> предсказание было слишком большим или слишком маленьким?
d_y_pred/d_z_out  -> насколько выходная сигмоида сейчас чувствительна?
d_z_out/d_h1      -> насколько h1 влияет на выход?
d_h1/d_z_h1       -> насколько скрытая сигмоида h1 чувствительна?
d_z_h1/d_w_11     -> насколько вес w_11 влияет на сумму z_h1?
```

Чтобы не писать длинную цепочку каждый раз, вводят короткое имя `delta_out`:

```text
delta_out = d_loss/d_y_pred * d_y_pred/d_z_out
```

А для скрытого нейрона:

```text
delta_h1 = delta_out * v1 * sigmoid'(z_h1)
```

После этого градиенты входящих весов скрытого нейрона считаются просто:

```text
d_loss/d_w_11 = delta_h1 * x1
d_loss/d_w_12 = delta_h1 * x2
d_loss/d_w_13 = delta_h1 * x3
```

То есть сложность спрятана в `delta_h1`, а дальше все снова похоже на обычный нейрон: дельта умножается на вход.


### Визуально: распределение ошибки по скрытым нейронам



`delta_out` не копируется одинаково во все скрытые нейроны. Он проходит через веса `v1`, `v2`, `v3` и умножается на производную активации соответствующего скрытого нейрона. Поэтому разные скрытые нейроны получают разные сигналы ошибки.


### 12.8. Численный пример: разобьем backpropagation на шаги

Теперь посчитаем все на одном объекте. Важно: дальше код специально разбит на маленькие куски. Каждый кусок отвечает на один вопрос.

План такой:

```text
1. выбрать объект;
2. сделать forward pass и сохранить промежуточные значения;
3. посчитать, как loss реагирует на предсказание;
4. получить delta_out;
5. посчитать градиенты выходного слоя;
6. передать ошибку на скрытые нейроны;
7. посчитать градиенты скрытого слоя;
8. сделать один шаг обновления весов.
```

Если держать в голове только идею “производная показывает чувствительность”, то каждый шаг становится довольно естественным.


<!-- derivative-friendly-extra -->
### Маленький словарь перед кодом

В коде ниже будут повторяться четыре слова.

```text
z      -> сумма до активации;
h      -> значение скрытого нейрона после sigmoid;
delta  -> сигнал ошибки для конкретного нейрона;
grad   -> градиент конкретного веса или bias.
```

Можно думать так:

```text
z и h появляются на forward pass;
delta и grad появляются на backward pass.
```


In [12]:
example_row = dataset[6]
inputs = row_to_vector(example_row)
target = example_row["umbrella"]

print(f"День: {example_row['day']}")
print(f"Входы: {inputs}")
print(f"Правильный ответ: {target}")


День: G
Входы: [0.3, 0.85, 0.9]
Правильный ответ: 1


Сначала выбираем один объект. Здесь это день `G`: влажность и ветер большие, правильный ответ равен `1`, то есть зонт нужен.

Backpropagation всегда начинается с обычного прямого прохода. Нам важно не только получить предсказание, но и сохранить промежуточные значения: `z` и активации скрытых нейронов. Они понадобятся при вычислении производных.


In [13]:
def sigmoid_derivative_from_output(value):
    return value * (1 - value)

hidden_z = []
hidden_outputs = []

for weights, bias in zip(network["hidden_weights"], network["hidden_biases"]):
    z = weighted_sum(inputs, weights, bias)
    hidden_z.append(z)
    hidden_outputs.append(sigmoid(z))

output_weights = network["output_weights"][0]
output_bias = network["output_biases"][0]
output_z = weighted_sum(hidden_outputs, output_weights, output_bias)
prediction = sigmoid(output_z)
loss = 0.5 * (prediction - target) ** 2

print("Скрытые нейроны:")
for index, (z, h) in enumerate(zip(hidden_z, hidden_outputs), start=1):
    print(f"  h{index}: z={z:.4f}, h=sigmoid(z)={h:.4f}")

print()
print(f"z_out = {output_z:.4f}")
print(f"prediction = {prediction:.4f}")
print(f"loss = {loss:.6f}")


Скрытые нейроны:
  h1: z=0.9000, h=sigmoid(z)=0.7109
  h2: z=2.3500, h=sigmoid(z)=0.9129
  h3: z=0.3500, h=sigmoid(z)=0.5866

z_out = 1.1400
prediction = 0.7577
loss = 0.029361


Теперь у нас есть все значения прямого прохода. Следующий шаг - понять, как loss меняется при изменении выхода сети.

Для квадратичной ошибки:

```text
loss = 0.5 * (y_pred - y_true)^2
```

производная по `y_pred` равна:

```text
d_loss/d_y_pred = y_pred - y_true
```

Но выход сети - это не сам `z_out`, а `sigmoid(z_out)`. Поэтому нужно еще умножить на производную сигмоиды.


In [14]:
d_loss_d_prediction = prediction - target
d_prediction_d_output_z = sigmoid_derivative_from_output(prediction)
output_delta = d_loss_d_prediction * d_prediction_d_output_z

print(f"d_loss/d_y_pred = {d_loss_d_prediction:.6f}")
print(f"d_y_pred/d_z_out = sigmoid'(z_out) = {d_prediction_d_output_z:.6f}")
print(f"delta_out = {output_delta:.6f}")


d_loss/d_y_pred = -0.242327
d_y_pred/d_z_out = sigmoid'(z_out) = 0.183605
delta_out = -0.044492


`delta_out` - это компактная запись сигнала ошибки на выходном нейроне.

Если `delta_out` отрицательная, то увеличение `z_out` уменьшит ошибку. В нашем примере правильный ответ равен 1, а сеть дала меньше 1, поэтому сети выгодно поднять выходное значение.

Теперь посчитаем градиенты весов выходного слоя. Каждый выходной вес умножает соответствующий скрытый нейрон:

```text
z_out = h1*v1 + h2*v2 + h3*v3 + b_out
```

Поэтому:

```text
d_loss/d_v1 = delta_out * h1
d_loss/d_v2 = delta_out * h2
d_loss/d_v3 = delta_out * h3
d_loss/d_b_out = delta_out
```


In [15]:
output_weight_gradients = []

for hidden_value in hidden_outputs:
    output_weight_gradients.append(output_delta * hidden_value)

output_bias_gradient = output_delta

for index, (hidden_value, gradient) in enumerate(
    zip(hidden_outputs, output_weight_gradients),
    start=1,
):
    print(
        f"d_loss/d_v{index} = delta_out * h{index} "
        f"= {output_delta:.6f} * {hidden_value:.6f} = {gradient:.6f}"
    )

print(f"d_loss/d_b_out = {output_bias_gradient:.6f}")


d_loss/d_v1 = delta_out * h1 = -0.044492 * 0.710950 = -0.031632
d_loss/d_v2 = delta_out * h2 = -0.044492 * 0.912934 = -0.040619
d_loss/d_v3 = delta_out * h3 = -0.044492 * 0.586618 = -0.026100
d_loss/d_b_out = -0.044492


Теперь ошибка должна пройти дальше назад, к скрытым нейронам.

Скрытый нейрон `h1` влияет на выход через вес `v1`. Поэтому сигнал ошибки для `h1` зависит от трех вещей:

```text
1. delta_out: какая ошибка возникла на выходе;
2. v1: насколько сильно h1 влияет на выход;
3. sigmoid'(z_h1): насколько h1 чувствителен к изменению своей входной суммы.
```

Формула:

```text
delta_h1 = delta_out * v1 * sigmoid'(z_h1)
```

То же самое делается для `h2` и `h3`.


In [16]:
hidden_deltas = []

for hidden_value, output_weight in zip(hidden_outputs, output_weights):
    hidden_derivative = sigmoid_derivative_from_output(hidden_value)
    hidden_delta = output_delta * output_weight * hidden_derivative
    hidden_deltas.append(hidden_delta)

for index, (output_weight, hidden_value, hidden_delta) in enumerate(
    zip(output_weights, hidden_outputs, hidden_deltas),
    start=1,
):
    derivative = sigmoid_derivative_from_output(hidden_value)
    print(
        f"delta_h{index} = delta_out * v{index} * sigmoid'(z_h{index}) "
        f"= {output_delta:.6f} * {output_weight:.6f} * {derivative:.6f} "
        f"= {hidden_delta:.6f}"
    )


delta_h1 = delta_out * v1 * sigmoid'(z_h1) = -0.044492 * 4.000000 * 0.205500 = -0.036573
delta_h2 = delta_out * v2 * sigmoid'(z_h2) = -0.044492 * 1.500000 * 0.079485 = -0.005305
delta_h3 = delta_out * v3 * sigmoid'(z_h3) = -0.044492 * -2.000000 * 0.242497 = 0.021579


Теперь у каждого скрытого нейрона есть свой сигнал ошибки: `delta_h1`, `delta_h2`, `delta_h3`.

Осталось получить градиенты весов, которые входят в скрытые нейроны. Здесь снова появляется простое правило: градиент входящего веса равен дельте нейрона, умноженной на соответствующий вход.

Например:

```text
d_loss/d_w_h1_cloudiness = delta_h1 * cloudiness
d_loss/d_w_h1_humidity   = delta_h1 * humidity
d_loss/d_w_h1_wind       = delta_h1 * wind
```

Bias не умножается на вход, поэтому его градиент просто равен дельте нейрона.


In [17]:
hidden_weight_gradients = []
hidden_bias_gradients = hidden_deltas[:]
input_names = ["cloudiness", "humidity", "wind"]

for hidden_delta in hidden_deltas:
    neuron_gradients = []
    for input_value in inputs:
        neuron_gradients.append(hidden_delta * input_value)
    hidden_weight_gradients.append(neuron_gradients)

for neuron_index, (hidden_delta, neuron_gradients) in enumerate(
    zip(hidden_deltas, hidden_weight_gradients),
    start=1,
):
    print(f"Скрытый нейрон h{neuron_index}")
    for input_name, input_value, gradient in zip(input_names, inputs, neuron_gradients):
        print(
            f"  d_loss/d_w_{input_name} = delta_h{neuron_index} * {input_name} "
            f"= {hidden_delta:.6f} * {input_value:.6f} = {gradient:.6f}"
        )
    print(f"  d_loss/d_b_h{neuron_index} = {hidden_delta:.6f}")
    print()


Скрытый нейрон h1
  d_loss/d_w_cloudiness = delta_h1 * cloudiness = -0.036573 * 0.300000 = -0.010972
  d_loss/d_w_humidity = delta_h1 * humidity = -0.036573 * 0.850000 = -0.031087
  d_loss/d_w_wind = delta_h1 * wind = -0.036573 * 0.900000 = -0.032915
  d_loss/d_b_h1 = -0.036573

Скрытый нейрон h2
  d_loss/d_w_cloudiness = delta_h2 * cloudiness = -0.005305 * 0.300000 = -0.001591
  d_loss/d_w_humidity = delta_h2 * humidity = -0.005305 * 0.850000 = -0.004509
  d_loss/d_w_wind = delta_h2 * wind = -0.005305 * 0.900000 = -0.004774
  d_loss/d_b_h2 = -0.005305

Скрытый нейрон h3
  d_loss/d_w_cloudiness = delta_h3 * cloudiness = 0.021579 * 0.300000 = 0.006474
  d_loss/d_w_humidity = delta_h3 * humidity = 0.021579 * 0.850000 = 0.018342
  d_loss/d_w_wind = delta_h3 * wind = 0.021579 * 0.900000 = 0.019421
  d_loss/d_b_h3 = 0.021579



Теперь градиенты найдены. Это еще не обновление весов, а только инструкция, как их менять.

Обновление делает оптимизатор. Для обычного градиентного спуска:

```text
parameter = parameter - learning_rate * gradient
```

Если градиент отрицательный, вес увеличится. Если градиент положительный, вес уменьшится. Ниже сделаем один маленький шаг и посмотрим, уменьшился ли loss.


In [18]:
def copy_network(network):
    return {
        "hidden_weights": [weights[:] for weights in network["hidden_weights"]],
        "hidden_biases": network["hidden_biases"][:],
        "output_weights": [weights[:] for weights in network["output_weights"]],
        "output_biases": network["output_biases"][:],
    }

learning_rate = 0.1
updated_network = copy_network(network)

for index, gradient in enumerate(output_weight_gradients):
    updated_network["output_weights"][0][index] -= learning_rate * gradient
updated_network["output_biases"][0] -= learning_rate * output_bias_gradient

for neuron_index, neuron_gradients in enumerate(hidden_weight_gradients):
    for input_index, gradient in enumerate(neuron_gradients):
        updated_network["hidden_weights"][neuron_index][input_index] -= learning_rate * gradient
    updated_network["hidden_biases"][neuron_index] -= learning_rate * hidden_bias_gradients[neuron_index]

new_prediction, _ = network_forward(inputs, updated_network)
new_loss = 0.5 * (new_prediction - target) ** 2

print(f"prediction до обновления:    {prediction:.6f}")
print(f"loss до обновления:          {loss:.6f}")
print(f"prediction после обновления: {new_prediction:.6f}")
print(f"loss после обновления:       {new_loss:.6f}")


prediction до обновления:    0.757673
loss до обновления:          0.029361
prediction после обновления: 0.761820
loss после обновления:       0.028365


После одного маленького шага loss уменьшился. Это не означает, что сеть уже обучена. Это означает, что для одного объекта и одного шага мы сдвинули параметры в правильную сторону.

В реальном обучении эти шаги повторяются много раз:

```text
1. взять объект или batch объектов;
2. сделать forward pass;
3. посчитать loss;
4. сделать backpropagation;
5. обновить веса;
6. повторить на следующих данных.
```

Один проход по всему обучающему датасету обычно называют эпохой.


### 12.9. Почему это называется обратным распространением

Ошибка не является веществом, которое буквально течет по сети. Речь идет о вычислении производных в обратном порядке.

При прямом проходе мы считаем значения:

```text
x -> z_h -> h -> z_out -> y_pred -> loss
```

При обратном проходе мы считаем чувствительности:

```text
loss -> y_pred -> z_out -> h -> z_h -> weights
```

Forward pass отвечает на вопрос:

```text
какое предсказание дает сеть с текущими весами?
```

Backpropagation отвечает на другой вопрос:

```text
как каждый вес повлиял на ошибку, и как его нужно изменить?
```

Если студент помнит только, что производная показывает изменение функции при маленьком изменении аргумента, этого уже достаточно для основной идеи. Backpropagation просто применяет эту идею много раз подряд: от `loss` к выходу, от выхода к скрытым нейронам, от скрытых нейронов к весам.


### 12.10. Итог

Соберем всю картину в одну последовательность.

```text
1. Архитектура задает, из каких слоев и связей состоит сеть.
2. Forward pass считает предсказание при текущих весах.
3. Loss превращает ошибку предсказания в одно число.
4. Производная показывает, как это число изменится при изменении параметра.
5. Backpropagation по правилу цепочки считает такие производные для всех весов.
6. Optimizer обновляет веса в сторону уменьшения loss.
7. Повторение этих шагов постепенно обучает сеть.
```

Самая важная мысль:

```text
обучение нейросети = много маленьких изменений весов,
каждое из которых выбирается по производной ошибки.
```

Backpropagation нужен, чтобы понять, какие именно маленькие изменения сделать.
